In [2]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import *

In [4]:
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
num_components = check_connectivity(edges, n_vertices)
if num_components > 1:
    edges, n_vertices, edge_weights = filter_to_largest_component(edges, n_vertices, edge_weights)

A, D, L = build_graph_matrices(edges, n_vertices)
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)

import networkx as nx
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

from sksparse.cholmod import cholesky as sparse_cholesky
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

Components: 1
  -> Graph is fully connected, safe to proceed

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 2056 (2056 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.40
Singletons: 7 (0.3%)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()
Interior coarse vertices: 2041 (99.3%)
coarse edges: 3260 (from 6594 fine edges)


# Single-level MC

In [8]:
import time

t0 = time.time()
Q_samples = monte_carlo_loop_tqdm(L_sigma, lambda_min, edges, n_vertices,
                                    gamma_in, gamma_out, N=1000, debug=False)
single_level_time = time.time() - t0

print(f"Single-level time: {single_level_time:.2f}s")
print(f"Mean Q: {Q_samples.mean():.6f}")
print(f"Std Q: {Q_samples.std():.6f}")

Monte Carlo: 100%|██████████| 1000/1000 [00:30<00:00, 32.32sample/s, mean Q=6.6159]


✓ Done — 1000 samples
  Mean Q : 6.615862
  Std Q  : 39.756764
Single-level time: 31.04s
Mean Q: 6.615862
Std Q: 39.756764


# My two-level code two-level code (N=300 paired, N=2000 coarse-only)

In [11]:
t0 = time.time()
result = run_paired_validation(setup, N=300)
paired_time = time.time() - t0

t0 = time.time()
Q_coarse_only_samples = np.array([
    run_coarse_only_sample(setup, seed=100000+n) for n in range(2000)
])
coarse_time = time.time() - t0

estimate = two_level_estimate(setup, result, N_coarse_only=2000)
own_code_time = paired_time + coarse_time

print(f"\nYour code — time: {own_code_time:.2f}s")
print(f"Estimate: {estimate['estimate']:.6f}")
print(f"Correlation: {result['correlation']:.4f}, Variance reduction: {result['variance_reduction']:.2f}x")

Paired samples: 100%|██████████| 300/300 [00:14<00:00, 20.39sample/s, Q_fine=7.2139, Q_coarse=10.9644] 



N = 300 paired samples
Q_fine   : mean=7.213877  var=2385.803171
Q_coarse : mean=10.964447  var=5566.866596
Q_fine - Q_coarse : mean=-3.750570  var=664.056741
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 3.59x


Coarse-only samples: 100%|██████████| 2000/2000 [00:52<00:00, 37.81sample/s, Q_coarse=11.5382]


Coarse-only base estimate (N=2000): 11.538190
Correction term mean (paired samples): -3.750570
Two-level estimate of E[Q_fine]: 7.787620
Direct fine-only mean (for comparison): 7.213877

Your code — time: 60.70s
Estimate: 7.787620
Correlation: 1.0000, Variance reduction: 3.59x


In [24]:
import numpy as np

# From your paired run
paired_diff_var = result['diff_samples'].var()
N_paired = 300  # adjust to whatever N you actually used

se_correction = np.sqrt(paired_diff_var / N_paired)

# From your coarse-only samples (need the actual array, not just the mean)
N_coarse_only = 2000  # adjust to whatever N you actually used
coarse_only_var = Q_coarse_only_samples.var()  # the array from your coarse-only loop

se_coarse = np.sqrt(coarse_only_var / N_coarse_only)

combined_se = np.sqrt(se_correction**2 + se_coarse**2)

print(f"SE from correction term: {se_correction:.6f}")
print(f"SE from coarse-only term: {se_coarse:.6f}")
print(f"Combined two-level SE (your code): {combined_se:.6f}")

SE from correction term: 1.487791
SE from coarse-only term: 2.141700
Combined two-level SE (your code): 2.607758


# Team's runner, same sample counts (300 paired, 2000 coarse-only)

In [14]:
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner

model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)

t0 = time.time()
mlmc_result = runner.run_fixed(samples_per_level=[2000, 300])
mlmc_time = time.time() - t0

print(f"\nMLMCRunner — time: {mlmc_time:.2f}s")
print(f"Estimate: {mlmc_result.estimate:.6f}")
print(f"SE: {mlmc_result.standard_error:.6f}")


MLMCRunner — time: 64.35s
Estimate: 27.909699
SE: 15.977836


In [22]:
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner

model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)

t0 = time.time()
mlmc_result = runner.run_fixed(samples_per_level=[3500,300])
mlmc_time = time.time() - t0

print(f"\nMLMCRunner — time: {mlmc_time:.2f}s")
print(f"Estimate: {mlmc_result.estimate:.6f}")
print(f"SE: {mlmc_result.standard_error:.6f}")


MLMCRunner — time: 90.95s
Estimate: 17.447573
SE: 9.217862
